# Stylometric & Lexical Author Classification

This notebook focuses on **Stylometry** (the study of linguistic style) to improve author identification. Instead of focus on semantics (what word means), we'll focus on the specific lexical choices and stylistic fingerprints of **Edgar Allan Poe (EAP)**, **H.P. Lovecraft (HPL)**, and **Mary Shelley (MWS)**.

### Experiments:

1. **Word-level TF-IDF (Baseline)**
2. **Character-level TF-IDF** (Captures morphology, suffixes, and punctuation)
3. **POS-tagged TF-IDF** (Captures purely grammatical style)
4. **Custom Stylometric Features** (Sentence length, vocabulary complexity, function words)
5. **Ensemble Pipeline**


In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import nltk
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import optuna
import warnings

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
warnings.filterwarnings('ignore')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [4]:
# Load data
df = pd.read_csv('spooky.csv')
df


,id,text,author
0,id26305,"This process, however, afforded me no means of...",EAP
1,id17569,It never once occurred to me that the fumbling...,HPL
2,id11008,"In his left hand was a gold snuff box, from wh...",EAP
3,id27763,How lovely is spring As we looked from Windsor...,MWS
4,id12958,"Finding nothing else, not even gold, the Super...",HPL
...,...,...,...
19574,id17718,"I could have fancied, while I looked at it, th...",EAP
19575,id08973,The lids clenched themselves together as if in...,EAP
19576,id05267,"Mais il faut agir that is to say, a Frenchman ...",EAP
19577,id17513,"For an item of news like this, it strikes us i...",EAP


In [6]:

df = df.dropna(subset=['text'])

le = LabelEncoder()
df['target'] = le.fit_transform(df['author'])

X_train, X_val, y_train, y_val = train_test_split(
    df['text'], df['target'], test_size=0.2, random_state=42
)

results = {}
print(f"Loaded {len(df)} samples with labels: {le.classes_}")

Loaded 19579 samples with labels: ['EAP' 'HPL' 'MWS']


## 1. Word-level TF-IDF (Baseline)

Default TF-IDF often performs well because it captures the specific vocabulary frequency of each author.


In [7]:
tfidf_word = TfidfVectorizer(ngram_range=(1, 1))
X_train_word = tfidf_word.fit_transform(X_train)
X_val_word = tfidf_word.transform(X_val)

lr_base = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_base.fit(X_train_word, y_train)
probs = lr_base.predict_proba(X_val_word)

results['Word TF-IDF Baseline'] = log_loss(y_val, probs)
print(f"Baseline Log-Loss: {results['Word TF-IDF Baseline']:.4f}")

Baseline Log-Loss: 0.5348


## 2. Character-level TF-IDF

Character n-grams are excellent for author identification because they capture sub-word patterns, punctuation, and morphological markers that authors use habitually without realizing it.


In [8]:
def objective_char(trial):
    n_range = trial.suggest_categorical('ngram_range', [(2, 4), (3, 5), (2, 5)])
    sublinear = trial.suggest_categorical('sublinear_tf', [True, False])

    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=n_range, sublinear_tf=sublinear,stop_words=None)
    X_tr = vectorizer.fit_transform(X_train)

    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
    score = cross_val_score(clf, X_tr, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_char = optuna.create_study(direction='maximize')
study_char.optimize(objective_char, n_trials=5)

params = study_char.best_params
tfidf_char = TfidfVectorizer(analyzer='char_wb', **params)
X_train_char = tfidf_char.fit_transform(X_train)
X_val_char = tfidf_char.transform(X_val)

lr_char = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_char.fit(X_train_char, y_train)
probs_char = lr_char.predict_proba(X_val_char)

results['Char TF-IDF (Optimized)'] = log_loss(y_val, probs_char)
print(f"Char-TFIDF Log-Loss: {results['Char TF-IDF (Optimized)']:.4f}")

[I 2026-04-15 08:01:41,584] A new study created in memory with name: no-name-eb65e026-07c3-4c60-849f-ec90cf00bd13
[I 2026-04-15 08:02:03,037] Trial 0 finished with value: -0.5519902295567773 and parameters: {'ngram_range': (3, 5), 'sublinear_tf': True}. Best is trial 0 with value: -0.5519902295567773.
[I 2026-04-15 08:02:32,129] Trial 1 finished with value: -0.5485470569829937 and parameters: {'ngram_range': (2, 5), 'sublinear_tf': False}. Best is trial 1 with value: -0.5485470569829937.
[I 2026-04-15 08:02:52,113] Trial 2 finished with value: -0.5519902295567773 and parameters: {'ngram_range': (3, 5), 'sublinear_tf': True}. Best is trial 1 with value: -0.5485470569829937.
[I 2026-04-15 08:03:17,937] Trial 3 finished with value: -0.5505049004318624 and parameters: {'ngram_range': (3, 5), 'sublinear_tf': False}. Best is trial 1 with value: -0.5485470569829937.
[I 2026-04-15 08:03:40,025] Trial 4 finished with value: -0.5519902295567773 and parameters: {'ngram_range': (3, 5), 'sublinear_

Char-TFIDF Log-Loss: 0.5041


## 3. Stylometric Feature Extraction

We create a custom class to extract features like word length, punctuation density, and function word ratios.


In [14]:
class StylometricExtractor(BaseEstimator, TransformerMixin):
    def fit(self, x, y=None):
        return self

    def transform(self, posts):
        features = []
        for post in posts:
            words = post.split()
            features.append([
                len(post),                   # Raw length
                len(words),                  # Word count
                np.mean([len(w) for w in words]) if words else 0, # Avg word length
                len(set(words)) / len(words) if words else 0,     # Lexical diversity
                post.count(','),             # Punctuation habits
                post.count(';'),
                post.count('!'),
                len([w for w in words if w.lower() in ['the', 'and', 'of', 'in', 'to']]) / len(words) if words else 0 # Stopword density
            ])
        return np.array(features)

extractor = StylometricExtractor()
X_train_sty = extractor.transform(X_train)
X_val_sty = extractor.transform(X_val)

# Must scale custom features
scaler = StandardScaler()
X_train_sty = scaler.fit_transform(X_train_sty)
X_val_sty = scaler.transform(X_val_sty)

lr_sty = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_sty.fit(X_train_sty, y_train)
results['Stylometric Features Only'] = log_loss(y_val, lr_sty.predict_proba(X_val_sty))
print(f"Stylometric Logic-Loss: {results['Stylometric Features Only']:.4f}")

Stylometric Logic-Loss: 1.0343


## 4. Part-of-Speech (POS) TF-IDF

This ignores the words and focuses on the sequence of grammar used.


In [10]:
def get_pos_tags(text):
    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)
    return " ".join([tag for word, tag in tags])

X_train_pos = X_train.apply(get_pos_tags)
X_val_pos = X_val.apply(get_pos_tags)

tfidf_pos = TfidfVectorizer(ngram_range=(1, 3))
X_train_pos_vec = tfidf_pos.fit_transform(X_train_pos)
X_val_pos_vec = tfidf_pos.transform(X_val_pos)

lr_pos = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_pos.fit(X_train_pos_vec, y_train)
results['POS TF-IDF'] = log_loss(y_val, lr_pos.predict_proba(X_val_pos_vec))
print(f"POS-TFIDF Log-Loss: {results['POS TF-IDF']:.4f}")

POS-TFIDF Log-Loss: 0.8608


## 5. Final Stylometric Union

We combine word-level, char-level, and manual stylometric features into one powerful feature set.


In [11]:
combined_features = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('char_tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)),
    ('stylometry', Pipeline([
        ('extract', StylometricExtractor()),
        ('scale', StandardScaler())
    ]))
])

X_train_final = combined_features.fit_transform(X_train)
X_val_final = combined_features.transform(X_val)

lr_final = LogisticRegression(C=2.0, multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_final.fit(X_train_final, y_train)
final_probs = lr_final.predict_proba(X_val_final)

results['Combined Stylometric Pipeline'] = log_loss(y_val, final_probs)

print("\n--- Final Rankings (Log-Loss) ---")
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name}: {score:.4f}")


--- Final Rankings (Log-Loss) ---
Combined Stylometric Pipeline: 0.4126
Char TF-IDF (Optimized): 0.5041
Word TF-IDF Baseline: 0.5348
POS TF-IDF: 0.8608
Stylometric Features Only: 1.0343


## 6. Optimization of the Final Pipeline

We now use **Optuna** to fine-tune the hyper-parameters of the best combination of features and the final classifier simultaneously. This includes:

- Tuning Word TF-IDF n-grams.
- Tuning Char TF-IDF n-grams.
- Tuning Logistic Regression regularization (`C`).


In [12]:
def objective_final(trial):
    # Features Optimization
    word_ngram = trial.suggest_categorical('word_ngram', [(1, 1), (1, 2)])
    char_ngram = trial.suggest_categorical('char_ngram', [(2, 4), (3, 5), (3, 6)])
    sublinear = trial.suggest_categorical('sublinear', [True, False])

    # Classifier Optimization
    C = trial.suggest_float('C', 0.1, 10.0, log=True)

    pipeline = Pipeline([
        ('union', FeatureUnion([
            ('word_tfidf', TfidfVectorizer(ngram_range=word_ngram, sublinear_tf=sublinear)),
            ('char_tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=char_ngram, sublinear_tf=sublinear)),
            ('stylometry', Pipeline([
                ('extract', StylometricExtractor()),
                ('scale', StandardScaler())
            ]))
        ])),
        ('clf', LogisticRegression(C=C, multi_class='multinomial', solver='lbfgs', max_iter=1000))
    ])

    # We use 3-fold cross-validation
    score = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='neg_log_loss').mean()
    return score

study_final = optuna.create_study(direction='maximize')
study_final.optimize(objective_final, n_trials=10)

best_p = study_final.best_params
print(f"Best Hyperparameters: {best_p}")

# Final Training with best params
final_pipeline_opt = Pipeline([
    ('union', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(ngram_range=best_p['word_ngram'], sublinear_tf=best_p['sublinear'])),
        ('char_tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=best_p['char_ngram'], sublinear_tf=best_p['sublinear'])),
        ('stylometry', Pipeline([
            ('extract', StylometricExtractor()),
            ('scale', StandardScaler())
        ]))
    ])),
    ('clf', LogisticRegression(C=best_p['C'], multi_class='multinomial', solver='lbfgs', max_iter=1000))
])

final_pipeline_opt.fit(X_train, y_train)
best_final_probs = final_pipeline_opt.predict_proba(X_val)

results['Optimized Stylometric Pipeline'] = log_loss(y_val, best_final_probs)
print(f"Optimized Pipeline Log-Loss: {results['Optimized Stylometric Pipeline']:.4f}")

[I 2026-04-15 08:05:00,676] A new study created in memory with name: no-name-5f381ab4-ae60-4a78-953c-c4ce1f29dbff
[I 2026-04-15 08:06:29,494] Trial 0 finished with value: -0.42322749565685774 and parameters: {'word_ngram': (1, 2), 'char_ngram': (2, 4), 'sublinear': False, 'C': 4.786193745114943}. Best is trial 0 with value: -0.42322749565685774.
[I 2026-04-15 08:07:57,063] Trial 1 finished with value: -0.41681062010823533 and parameters: {'word_ngram': (1, 2), 'char_ngram': (2, 4), 'sublinear': False, 'C': 6.782036105852013}. Best is trial 1 with value: -0.41681062010823533.
[I 2026-04-15 08:08:58,129] Trial 2 finished with value: -0.4178264746955236 and parameters: {'word_ngram': (1, 1), 'char_ngram': (2, 4), 'sublinear': False, 'C': 7.164063119221182}. Best is trial 1 with value: -0.41681062010823533.
[I 2026-04-15 08:09:58,609] Trial 3 finished with value: -0.5158674807433318 and parameters: {'word_ngram': (1, 2), 'char_ngram': (3, 5), 'sublinear': True, 'C': 0.7510141022450294}. Be

Best Hyperparameters: {'word_ngram': (1, 2), 'char_ngram': (3, 5), 'sublinear': True, 'C': 9.901446784429009}
Optimized Pipeline Log-Loss: 0.3763


## 7. Model Calibration (Final Polish)

Log-loss is sensitive to probability accuracy. We will use **CalibratedClassifierCV** to ensure the model outputs reliable probabilities. This often improves the score by another 0.01-0.02.


In [13]:
from sklearn.calibration import CalibratedClassifierCV

# Use the best pipeline and wrap it with calibration
calibrated_pipeline = CalibratedClassifierCV(final_pipeline_opt, method='sigmoid', cv=3)
calibrated_pipeline.fit(X_train, y_train)

calibrated_probs = calibrated_pipeline.predict_proba(X_val)
results['Calibrated Optimized Pipeline'] = log_loss(y_val, calibrated_probs)

print(f"Calibrated Pipeline Log-Loss: {results['Calibrated Optimized Pipeline']:.4f}")

# Update final visualization
print("\n--- Final Rankings (Log-Loss) ---")
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name}: {score:.4f}")

Calibrated Pipeline Log-Loss: 0.4190

--- Final Rankings (Log-Loss) ---
Optimized Stylometric Pipeline: 0.3763
Combined Stylometric Pipeline: 0.4126
Calibrated Optimized Pipeline: 0.4190
Char TF-IDF (Optimized): 0.5041
Word TF-IDF Baseline: 0.5348
POS TF-IDF: 0.8608
Stylometric Features Only: 1.0343
